In [ ]:
# runs in Chameleon Jupyter environment
from chi import server, context, lease, network
import chi, os, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

username = os.getenv("USER")
PROJECT_SUFFIX = "proj13"
LEASE_NAME = f"lease-serve-model-{username}-{PROJECT_SUFFIX}"
SERVER_NAME = f"node-serve-model-{username}-{PROJECT_SUFFIX}"
FLAVOR_NAME = "m1.medium"
IMAGE_NAME = "CC-Ubuntu24.04"


In [ ]:
# runs in Chameleon Jupyter environment
l = lease.Lease(LEASE_NAME, duration=datetime.timedelta(hours=8))
l.add_flavor_reservation(id=chi.server.get_flavor_id(FLAVOR_NAME), amount=1)
l.submit(idempotent=True)
l.show()


In [ ]:
# runs in Chameleon Jupyter environment
s = server.Server(
    SERVER_NAME,
    image_name=IMAGE_NAME,
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)


In [ ]:
# runs in Chameleon Jupyter environment
s.associate_floating_ip()
s.refresh()
s.check_connectivity()


In [ ]:
# runs in Chameleon Jupyter environment
security_groups = [
    {"name": "allow-ssh", "port": 22, "description": "Enable SSH traffic on TCP port 22"},
    {"name": "allow-8888", "port": 8888, "description": "Enable TCP port 8888 (used by Jupyter)"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        "name": sg["name"],
        "description": sg["description"],
    })
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"updated security groups: {[sg['name'] for sg in security_groups]}")
s.refresh()
s.show(type="widget")


In [ ]:
# runs in Chameleon Jupyter environment
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
s.execute("sudo apt-get install -y docker-compose-plugin")


In [ ]:
# runs in Chameleon Jupyter environment
REPO_URL = "https://github.com/yanghao13111/SmartQueue.git"
s.execute(f"git clone {REPO_URL} ~/smartqueue")


In [ ]:
# runs in Chameleon Jupyter environment
s.execute("python3 -m pip install torch --index-url https://download.pytorch.org/whl/cpu -q")
s.execute("cd ~/smartqueue/serving && python3 models/create_model.py")


Open an SSH session from your local terminal:

```
ssh -i ~/.ssh/id_rsa_chameleon cc@A.B.C.D
```

Replace `A.B.C.D` with the floating IP shown above.

In [ ]:
# runs on the VM over SSH
# cd ~/smartqueue/serving && docker compose -f docker/docker-compose-model.yaml up --build -d


In [ ]:
# runs on the VM over SSH
# docker exec jupyter jupyter server list


Paste the token URL into a browser tab, replacing `127.0.0.1` with the VM floating IP.

Then open notebooks `1_baseline_pytorch.ipynb` through `5_static_quantize.ipynb` from the `work/` directory.

In [ ]:
# runs in Chameleon Jupyter environment
# delete the VM when finished to keep costs low
s.delete()
